In [ ]:
# CELL 1 — Clean install (no conflicts)
!apt install ffmpeg -y -q

# Uninstall conflicting packages first
!pip uninstall -y gtts click transformers -q

# Install in correct order
!pip install "click>=8.2.1" -q
!pip install "transformers>=4.41.0" -q
!pip install openai-whisper SpeechRecognition pydub -q

print(" All installed successfully!")

In [ ]:
from pydub import AudioSegment
audio = AudioSegment.from_mp3("/content/Meeting Audio.mp3")
audio.export("/content/Speech.wav", format="wav")
print(" Conversion done!")

In [ ]:
import whisper
model = whisper.load_model("base")
result = model.transcribe("/content/Speech.wav")
transcript = result["text"]
print(" Transcript:\n", transcript)

In [ ]:
# CELL 4 — Summarize (works with transformers 5.x)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

# Tokenize input (BART has 1024 token limit)
inputs = tokenizer(transcript, return_tensors="pt", max_length=1024, truncation=True)

# Generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(" Summary:\n", summary_text)

In [ ]:
# CELL 5 — Extract Action Items
import spacy

# Download the model if not already present
!python -m spacy download en_core_web_sm -q

nlp = spacy.load("en_core_web_sm")

def extract_action_items(text):
    sentences = text.split(".")
    action_items = []
    action_verbs = ["will", "should", "must", "needs to", "going to", "has to", "to"]

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if any(verb in sentence.lower() for verb in action_verbs):
            doc = nlp(sentence)
            assignee = next((ent.text for ent in doc.ents if ent.label_ == "PERSON"), "Unknown")
            deadline = next((ent.text for ent in doc.ents if ent.label_ == "DATE"), None)
            action_items.append({
                "action": sentence,
                "assignee": assignee,
                "deadline": deadline
            })
    return action_items

items = extract_action_items(transcript)

print(" Extracted Action Items:\n")
if not items:
    print("No action items found.")
else:
    for i, item in enumerate(items, 1):
        print(f"{i}. Action   : {item['action']}")
        print(f"   Assigned : {item['assignee']}")
        print(f"   Deadline : {item['deadline'] if item['deadline'] else 'Not mentioned'}")
        print()

In [ ]:
from google.colab import files
files.upload()  # A file picker will open → select your credentials.json

In [ ]:
# CELL 6 — Google Calendar Event Creator (Fixed)
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib -q

import os
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/calendar']

def get_calendar_service():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file(
            '/content/credentials.json.json',   # ✅ fixed filename (was .json.json)
            SCOPES
        )

        # ✅ New way — works in Colab
        import google_auth_oauthlib.flow as flow_module
        flow.redirect_uri = 'urn:ietf:wg:oauth:2.0:oob'
        auth_url, _ = flow.authorization_url(prompt='consent')

        print("👉 Open this URL in your browser:\n")
        print(auth_url)
        print()
        code = input("📋 Paste the authorization code here: ")
        flow.fetch_token(code=code)
        creds = flow.credentials

        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    return build('calendar', 'v3', credentials=creds)

def create_event(summary, description, start_time, end_time, attendees_emails=[]):
    service = get_calendar_service()
    event = {
        'summary': summary,
        'description': description,
        'start': {'dateTime': start_time, 'timeZone': 'Asia/Kolkata'},
        'end':   {'dateTime': end_time,   'timeZone': 'Asia/Kolkata'},
        'attendees': [{'email': e} for e in attendees_emails],
    }
    event = service.events().insert(
        calendarId='primary',
        body=event,
        sendUpdates='all'
    ).execute()
    print('✅ Event created:', event.get('htmlLink'))

# ── Create event from meeting summary ──
create_event(
    summary     = "Meeting Follow-up",
    description = summary_text,
    start_time  = "2026-05-29T15:00:00+05:30",
    end_time    = "2026-05-29T16:00:00+05:30",
    attendees_emails = []
)